In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import seaborn as sns
import numpy as np
import matplotlib.colors as colors
import matplotlib.cm as cm
from tqdm import tqdm

%config InlineBackend.figure_format = "retina"

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/RNAseq' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/RNAseq' 
savepath_fig.mkdir(parents=True, exist_ok=True)

width_two_col = 183 / 25.4 # mm to inch

In [ ]:
receptor_genes = ['GluClalpha', 'Rdl', 'ort', 'HisCl1', 'Grd']
histamine_genes = ['CarT', 'Hdc', 'lovit', 't']

In [ ]:
# Davis data (https://doi.org/10.7554/eLife.50901)
# Downloaded GSE116969_dataTable7b.genes_x_cells_p_expression.modeled_genes.txt from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116969
df_davis = pd.read_csv(inputpath_data / 'Davis_etal/GSE116969_dataTable7b.genes_x_cells_p_expression.modeled_genes.txt', sep=r'\s+')

incl = ['R1-6', 'L1', 'L2', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9', 'T4', 'T5']
plotting = df_davis[incl].copy().reset_index().rename(columns={'index':'gene_name'})

GOI_avg_davis = plotting.query("gene_name in ['Hdc', 'CarT', 't', 'CG45782', 'GluClalpha', 'Rdl', 'HisCl1', 'ort', 'Grd']") # CG45782 = lovit
GOI_avg_davis['gene_name'] = GOI_avg_davis.gene_name.replace({'CG45782':'lovit'})
GOI_avg_davis = GOI_avg_davis.rename(columns={'gene_name':'gene'})
GOI_avg_davis = GOI_avg_davis.reset_index(drop=True)

In [ ]:
# Hoermann data (https://doi.org/10.1242/dev.186296)
# Used preprocessed (in R) version of these data: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE147987, provided by original author
df_hoermann = pd.read_csv(inputpath_data / 'Hoermann_etal/GSE147987_Hoermann_preprocessed_data.csv', sep=',')

plotting_hoermann = df_hoermann.query("gene in ['GluClalpha', 'Rdl', 'HisCl1', 'ort', 'Grd']") 
plotting_hoermann_avg = pd.DataFrame(plotting_hoermann.groupby(['gene', 'emin']).expression.mean()).reset_index()

# Calculate the mean value of the subtypes
cell_types = ['T4', 'T5']
avg = {}
for cell_type in cell_types:
    avg[cell_type] = plotting_hoermann.query("emin.str.contains(@cell_type)", engine='python').groupby('gene').expression.mean()

GOI_avg_hoermann = pd.DataFrame(avg).reset_index()
GOI_avg_hoermann

In [ ]:
# Ozel data (https://doi.org/10.1038/s41586-020-2879-3)
# Downloaded GSE142787_Log_normalized_average_expression.xlsx from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE142787
ozel_data = {}
for excel_sheet in tqdm(['P15_average_expression', 'P30_average_expression', 'P40_average_expression', 'P50_average_expression', 'P70_average_expression', 'Adult_average_expression']):
    temp = pd.read_excel(inputpath_data / 'Ozel_etal/GSE142787_Log_normalized_average_expression.xlsx', index_col=0, sheet_name=excel_sheet)
    temp = temp.copy()
    temp = temp.reset_index().rename(columns={'index':'gene'})
    temp = temp.query("gene in ['Hdc', 'CG9317', 't', 'CG45782', 'GluClalpha', 'Rdl', 'HisCl1', 'ort', 'Grd']") # CG45782 = lovit, CG9317 = CarT
    ozel_data[excel_sheet] = temp

In [ ]:
ozel = ozel_data.copy()

ozel['P15_average_expression'] = ozel['P15_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
ozel['P30_average_expression'] = ozel['P30_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
ozel['P40_average_expression'] = ozel['P40_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
ozel['P50_average_expression'] = ozel['P50_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
ozel['P70_average_expression'] = ozel['P70_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
ozel['Adult_average_expression'] = ozel['Adult_average_expression'].reset_index(drop=True)[['gene', 109, 108, 234, 235, 164, 165, 3, 142, 33, 140, 118, 125, 126, 227, 228]]
for subset in ozel:
    ozel[subset]['stage'] = subset
    ozel[subset] = ozel[subset].rename(columns={109:'L1', 108:'L2',234:'T4T5ab', 235:'T4T5cd', 164:'Mi4', 165:'C3', 3:'R1-8', 142:'Mi1', 33:'Mi9', 140:'Tm1', 118:'Tm2', 126:'Tm3', 125:'Tm4', 227:'Tm9v', 228:'Tm9d'})

In [ ]:
plotting_ozel = pd.concat(ozel).reset_index(drop=True)
plotting_ozel['stage'] = plotting_ozel['stage'].str.split('_', expand=True)[0]
plotting_ozel = pd.melt(plotting_ozel, id_vars=['stage', 'gene'], value_vars=['L1', 'L2', 'T4T5ab', 'T4T5cd', 'Mi4', 'C3', 'R1-8', 'Mi1', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9v', 'Tm9d'], var_name='celltype', value_name='expression')

plotting_ozel_development = pd.DataFrame(plotting_ozel.query("stage != 'Adult'").groupby(['celltype','gene']).expression.mean())
plotting_ozel_adult = pd.DataFrame(plotting_ozel.query("stage == 'Adult'").groupby(['celltype','gene']).expression.mean())

GOI_avg_ozel_dev = plotting_ozel_development.reset_index().pivot(index='gene', columns='celltype', values='expression').reset_index()
GOI_avg_ozel_dev.gene = GOI_avg_ozel_dev.gene.replace({'CG45782':'lovit', 'CG9317':'CarT'})

GOI_avg_ozel_adult = plotting_ozel_adult.reset_index().pivot(index='gene', columns='celltype', values='expression').reset_index()
GOI_avg_ozel_adult.gene = GOI_avg_ozel_adult.gene.replace({'CG45782':'lovit', 'CG9317':'CarT'})


In [ ]:
# Yoo Data (https://doi.org/10.1016/j.cub.2023.08.020)
# Downloaded expr_V1.1a.tsv.gz from https://zenodo.org/records/17932486 
yoo = pd.read_csv(inputpath_data / 'Yoo_etal/expr_V1.1a.tsv', sep="\t", memory_map = True)
yoo

In [ ]:
yoo_genes = yoo.query("set == 'DGRP' and gene in ['Hdc', 'CarT', 't', 'lovit', 'GluClalpha', 'Rdl', 'HisCl1', 'ort', 'Grd'] and type2 in ['R1.6', 'C3', 'Mi4', 'Tm2', 'Tm1', 'Tm4', 'Tm9', 'Mi1', 'Tm3', 'Mi9', 'T4.T5', 'L1', 'L2']")
yoo_genes['type'] = yoo_genes.subtype2.replace({'L1':'L1', 'L2':'L2', 'T4a':'T4', 'T4b':'T4', 'T4c':'T4', 'T4d':'T4', 'T5a':'T5', 'T5b':'T5', 'T5c':'T5', 'T5d':'T5'})
yoo_avg = pd.DataFrame(yoo_genes.groupby(['type', 'gene']).expr.mean()).reset_index()

In [ ]:
# Receptor genes
yoo_wide_rec = yoo_avg.query("type in ['L1', 'L2', 'T4', 'T5'] and gene in ['GluClalpha', 'Rdl', 'HisCl1', 'ort', 'Grd']").pivot(index='gene', columns='type', values='expr')
GOI_avg_yoo_rec = yoo_wide_rec.copy().reset_index()
GOI_avg_yoo_rec['gene'] = pd.Categorical(GOI_avg_yoo_rec.gene.to_list(), categories=receptor_genes, ordered=True)
GOI_avg_yoo_rec = GOI_avg_yoo_rec.sort_values('gene').set_index('gene')

GOI_avg_hoermann_rec = GOI_avg_hoermann.query("gene in @receptor_genes")
GOI_avg_hoermann_rec['gene'] = pd.Categorical(GOI_avg_hoermann_rec.gene.to_list(), categories=receptor_genes, ordered=True)
GOI_avg_hoermann_rec = GOI_avg_hoermann_rec.sort_values('gene').set_index('gene')

GOI_avg_ozel_dev_rec = GOI_avg_ozel_dev.query("gene in @receptor_genes")[['gene','L1', 'L2', 'T4T5ab', 'T4T5cd']]
GOI_avg_ozel_dev_rec['gene'] = pd.Categorical(GOI_avg_ozel_dev_rec.gene.to_list(), categories=receptor_genes, ordered=True)
GOI_avg_ozel_dev_rec = GOI_avg_ozel_dev_rec.sort_values('gene').set_index('gene')

GOI_avg_ozel_adult_rec = GOI_avg_ozel_adult.query("gene in @receptor_genes")[['gene','L1', 'L2', 'T4T5ab', 'T4T5cd']]
GOI_avg_ozel_adult_rec['gene'] = pd.Categorical(GOI_avg_ozel_adult_rec.gene.to_list(), categories=receptor_genes, ordered=True)
GOI_avg_ozel_adult_rec = GOI_avg_ozel_adult_rec.sort_values('gene').set_index('gene')

GOI_avg_davis_rec = GOI_avg_davis.query("gene in @receptor_genes")[['gene','L1', 'L2', 'T4', 'T5']]
GOI_avg_davis_rec['gene'] = pd.Categorical(GOI_avg_davis_rec.gene.to_list(), categories=receptor_genes, ordered=True)
GOI_avg_davis_rec = GOI_avg_davis_rec.sort_values('gene').set_index('gene')


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(1, 1), gridspec_kw={'height_ratios': [1, 20], 'hspace':0.2}, squeeze=True)

bw = colors.LinearSegmentedColormap.from_list(colors=[(1, 1, 1), (0, 0, 0)], name='bw')
sns.heatmap(np.log1p(GOI_avg_yoo_rec.loc[['GluClalpha', 'Rdl', 'HisCl1', 'ort']]), vmin=0, cmap=bw, cbar=None, ax=ax[1]);
cbar = fig.colorbar(mappable=cm.ScalarMappable(norm=colors.Normalize(vmin=0, vmax=np.log1p(GOI_avg_yoo_rec).values.max()), cmap=bw), cax=ax[0], location='top')
cbar_ax0 = ax[0].figure.get_axes()[-1] 
ax[0].set_aspect('auto')
cbar.outline.set_linewidth(0.5)
cbar.set_ticks([0, 1, 2, 3])

ax[1].set_ylabel('', visible=False);
ax[1].set_xlabel('', visible=False);
ax[1].set_yticklabels(ax[1].get_yticklabels(), rotation=0);

figtitle = 'yoo_RNAseq_receptors.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
fig, ax = plt.subplots(1,5, figsize=(width_two_col, 2.5), gridspec_kw={'width_ratios':[2,4,4,4,4]})

bw = colors.LinearSegmentedColormap.from_list(colors=[(1, 1, 1), (0, 0, 0)], name='bw')
sns.heatmap(np.log1p(GOI_avg_hoermann_rec.loc[['GluClalpha', 'Rdl', 'Grd', 'HisCl1', 'ort']]), vmin=0,cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04, 'aspect':10}, square=True, ax=ax[0]);
cbar_ax0 = ax[0].figure.get_axes()[-1] 
sns.heatmap(np.log1p(GOI_avg_yoo_rec.loc[['GluClalpha', 'Rdl', 'Grd', 'HisCl1', 'ort']]), vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[1]);
cbar_ax1 = ax[1].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_ozel_dev_rec.loc[['GluClalpha', 'Rdl', 'Grd', 'HisCl1', 'ort']], vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[2]);
cbar_ax2 = ax[2].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_ozel_adult_rec.loc[['GluClalpha', 'Rdl', 'Grd', 'HisCl1', 'ort']], vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[3]);
cbar_ax3 = ax[3].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_davis_rec.loc[['GluClalpha', 'Rdl', 'Grd', 'HisCl1', 'ort']], vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[4]);
cbar_ax4 = ax[4].figure.get_axes()[-1] 

for subplot in [ax[0], ax[1], ax[2], ax[3], ax[4]]:
    subplot.set_ylabel('', visible=False);
    subplot.set_xlabel('', visible=False);
    subplot.set_xticklabels(rotation=90, labels=subplot.get_xticklabels()); 
    if subplot != ax[0]:
        subplot.axes.get_yaxis().set_visible(False)

ax[0].set_title('Hörmann et al. (2020)\nGSE147987\nP24–P72', pad=30);
ax[1].set_title('Yoo et al. (2023)\nGSE156455\nP24–P96', pad=30);
ax[2].set_title('Özel et al. (2020)\nGSE142787\nP15–P70', pad=30);
ax[3].set_title('Özel et al. (2020)\nGSE142787\nAdult', pad=30);
ax[4].set_title('Davis et al. (2020)\nGSE116969\nAdult', pad=30);

for cb in [cbar_ax0, cbar_ax1, cbar_ax2, cbar_ax3, cbar_ax4]:
    cb.spines['outline'].set_visible(True)
    cb.spines['outline'].set_linewidth(0.5)
cbar_ax0.set_xlabel('loge(mean(UMI) + 1)', labelpad=2) 
cbar_ax1.set_xlabel('loge(mean(TP10K) + 1)', labelpad=2)
cbar_ax2.set_xlabel('mean(loge(UMI + 1))', labelpad=2)
cbar_ax3.set_xlabel('mean(loge(UMI + 1))', labelpad=2)
cbar_ax4.set_xlabel('Expression probability (AU)', labelpad=2)
cbar_ax4.set_xticks([0,0.5,1])

figtitle = 'all_RNAseq_receptors.pdf'
plt.savefig(savepath_fig / figtitle)

In [ ]:
# Histamine synthesis/transport genes
yoo_wide_hist = yoo_avg.query("type in ['R1.6', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9', 'T4', 'T5'] and gene in @histamine_genes").pivot(index='gene', columns='type', values='expr')
GOI_avg_yoo_hist = yoo_wide_hist.copy().reset_index()
GOI_avg_yoo_hist['gene'] = pd.Categorical(GOI_avg_yoo_hist.gene.to_list(), categories=histamine_genes, ordered=True)
GOI_avg_yoo_hist = GOI_avg_yoo_hist.sort_values('gene').set_index('gene')
GOI_avg_yoo_hist = GOI_avg_yoo_hist[['R1.6', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9', 'T4', 'T5']]

GOI_avg_ozel_dev_hist = GOI_avg_ozel_dev.query("gene in @histamine_genes")[['gene', 'R1-8', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9v', 'Tm9d', 'T4T5ab', 'T4T5cd']]
GOI_avg_ozel_dev_hist['gene'] = pd.Categorical(GOI_avg_ozel_dev_hist.gene.to_list(), categories=histamine_genes, ordered=True)
GOI_avg_ozel_dev_hist = GOI_avg_ozel_dev_hist.sort_values('gene').set_index('gene')

GOI_avg_ozel_adult_hist = GOI_avg_ozel_adult.query("gene in @histamine_genes")[['gene', 'R1-8', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9v', 'Tm9d', 'T4T5ab', 'T4T5cd']]
GOI_avg_ozel_adult_hist['gene'] = pd.Categorical(GOI_avg_ozel_adult_hist.gene.to_list(), categories=histamine_genes, ordered=True)
GOI_avg_ozel_adult_hist = GOI_avg_ozel_adult_hist.sort_values('gene').set_index('gene')

GOI_avg_davis_hist = GOI_avg_davis.query("gene in @histamine_genes")[['gene', 'R1-6', 'C3', 'Mi1', 'Mi4', 'Mi9', 'Tm1', 'Tm2', 'Tm3', 'Tm4', 'Tm9', 'T4', 'T5']]
GOI_avg_davis_hist['gene'] = pd.Categorical(GOI_avg_davis_hist.gene.to_list(), categories=histamine_genes, ordered=True)
GOI_avg_davis_hist = GOI_avg_davis_hist.sort_values('gene').set_index('gene')

In [ ]:
fig, ax = plt.subplots(1,4, figsize=(width_two_col, 2.5), gridspec_kw={'width_ratios':[12,13,13,12]})

bw = colors.LinearSegmentedColormap.from_list(colors=[(1, 1, 1), (0, 0, 0)], name='bw')
sns.heatmap(np.log1p(GOI_avg_yoo_hist), vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[0]);
cbar_ax0 = ax[0].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_ozel_dev_hist, vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[1]);
cbar_ax1 = ax[1].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_ozel_adult_hist, vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[2]);
cbar_ax2 = ax[2].figure.get_axes()[-1] 
sns.heatmap(GOI_avg_davis_hist, vmin=0, cmap=bw, cbar_kws={'location':'top', 'fraction':0.046, 'pad':0.04}, square=True, ax=ax[3]);
cbar_ax3 = ax[3].figure.get_axes()[-1] 

for subplot in [ax[0], ax[1], ax[2], ax[3]]:
    subplot.set_ylabel('', visible=False);
    subplot.set_xlabel('', visible=False);
    subplot.set_xticklabels(rotation=90, labels=subplot.get_xticklabels()); 
    if subplot != ax[0]:
        subplot.axes.get_yaxis().set_visible(False)

ax[0].set_title('Yoo et al. (2023)\nGSE156455\nP24–P96', pad=30);
ax[1].set_title('Özel et al. (2020)\nGSE142787\nP15–P70', pad=30);
ax[2].set_title('Özel et al. (2020)\nGSE142787\nAdult', pad=30);
ax[3].set_title('Davis et al. (2020)\nGSE116969\nAdult', pad=30);

for cb in [cbar_ax0, cbar_ax1, cbar_ax2, cbar_ax3]:
    cb.spines['outline'].set_visible(True)
    cb.spines['outline'].set_linewidth(0.5)
cbar_ax0.set_xlabel('loge(mean(TP10K) + 1)', labelpad=2)
cbar_ax1.set_xlabel('mean(loge(UMI + 1))', labelpad=2)
cbar_ax2.set_xlabel('mean(loge(UMI + 1))', labelpad=2)
cbar_ax3.set_xlabel('Expression probability (AU)', labelpad=2)
cbar_ax3.set_xticks([0,0.5,1])

figtitle = 'all_RNAseq_histamine.pdf'
plt.savefig(savepath_fig / figtitle)